# **Modelo Preditivo Imobiliário**

### Precificação de imóveis residenciais em King County

---

<table width="100%" cellpadding="12">
  <tr><th width="27%" align="left"><big>Identificação</big></th><th align="left"><big>Informação</big></th></tr>
  <tr><td><big><b>Curso</b></big></td><td><big>Desenvolvimento em IA para Análise Preditiva</big></td></tr>
  <tr><td><big><b>Projeto</b></big></td><td><big>Projeto Avaliativo Final — Módulo I</big></td></tr>
  <tr><td><big><b>Professor</b></big></td><td><big>Maurício Costa Cordeiro</big></td></tr>
  <tr><td><big><b>Aluno</b></big></td><td><big>Ricardo Pasquali dos Reis</big></td></tr>
  <tr><td><big><b>Dataset</b></big></td><td><big>King County House Sales — Opção A</big></td></tr>
  <tr><td><big><b>Ambiente</b></big></td><td><big>Google Colab</big></td></tr>
  <tr><td><big><b>Problema</b></big></td><td><big>Regressão supervisionada</big></td></tr>
  <tr><td><big><b>Variável-alvo</b></big></td><td><big><code>price</code>, preço de venda em dólares</big></td></tr>
</table>

---

## **Problema preditivo declarado**

Este projeto busca **prever a variável-alvo `price`**, um valor numérico contínuo que representa o preço de venda do imóvel em dólares, a partir de características físicas, temporais e de localização conhecidas antes da transação.

Isso importa para o negócio porque uma estimativa inicial consistente permite **priorizar imóveis, selecionar comparáveis e identificar preços possivelmente superavaliados ou subavaliados**. Uma diferença de dezenas de milhares de dólares afeta diretamente negociação, entrada, financiamento e exposição ao risco; por isso, o resultado será apresentado com métricas de erro e limites de uso, e não como um preço final automático.

> **Objetivo operacional:** produzir uma referência inicial de preço para apoiar a triagem imobiliária. A previsão não substitui vistoria, análise de comparáveis ou laudo, especialmente nos imóveis de maior valor.


# **1. Preparação do ambiente acadêmico**

A preparação limita-se às importações, às duas dependências geoespaciais, à definição dos caminhos e ao download validado da base pública. As funções analíticas são apresentadas somente nas fases em que passam a ser utilizadas.


In [ ]:
import hashlib, importlib.util, json, os, platform, shutil, subprocess, sys, tempfile, urllib.request
from pathlib import Path

faltantes = [pacote for modulo, pacote in {"h3": "h3>=4.1", "plotly": "plotly>=5.18", "jinja2": "jinja2>=3.1"}.items() if importlib.util.find_spec(modulo) is None]
if faltantes:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *faltantes], check=True)

import h3, joblib, matplotlib, matplotlib.pyplot as plt, numpy as np, pandas as pd, plotly
import scipy, seaborn as sns, sklearn
from IPython.display import FileLink, display
from sklearn.inspection import permutation_importance
from sklearn.model_selection import GroupShuffleSplit

SEED, TEST_SIZE, CV_FOLDS, N_JOBS, TARGET = 42, 0.20, 5, 1, "price"
em_colab = importlib.util.find_spec("google") is not None and importlib.util.find_spec("google.colab") is not None
ROOT = Path(os.environ["PROJETO_ROOT"]) if os.environ.get("PROJETO_ROOT") else (Path("/content/modelo_preditivo_imobiliario") if em_colab else Path(tempfile.gettempdir()) / "modelo_preditivo_imobiliario")
RAW_DATA = ROOT / "data/raw/kc_house_data.csv"
PROCESSED_DATA, FINAL_DATA = ROOT / "data/processed/kc_house_processed.csv", ROOT / "data/final/kc_house_final.csv"
FIGURES_DIR, MAPS_DIR, MODEL_DIR = ROOT / "outputs/figures", ROOT / "outputs/maps", ROOT / "models/v1"
for pasta in [RAW_DATA.parent, PROCESSED_DATA.parent, FINAL_DATA.parent, FIGURES_DIR, MAPS_DIR, MODEL_DIR]:
    pasta.mkdir(parents=True, exist_ok=True)

def baixar_base():
    assinatura = "aa501a2b18e0d0a288a0fdeb043b2c89d6c7e2ead1f9588165537d4e4e12b9df"
    if RAW_DATA.exists() and hashlib.sha256(RAW_DATA.read_bytes()).hexdigest() == assinatura:
        return
    erros = []
    for url in [
        "https://drive.usercontent.google.com/download?id=1XMw1aKYqfFA13AbOzd9zRElv216bCxqZ&export=download&confirm=t",
        "https://drive.google.com/uc?export=download&id=1XMw1aKYqfFA13AbOzd9zRElv216bCxqZ",
    ]:
        parcial = RAW_DATA.with_suffix(".part")
        try:
            req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req, timeout=120) as origem, parcial.open("wb") as destino:
                shutil.copyfileobj(origem, destino, 1024 * 1024)
            obtida = hashlib.sha256(parcial.read_bytes()).hexdigest()
            if obtida != assinatura:
                raise RuntimeError(f"SHA-256 inesperado: {obtida}")
            parcial.replace(RAW_DATA)
            return
        except Exception as erro:
            parcial.unlink(missing_ok=True)
            erros.append(f"{type(erro).__name__}: {erro}")
    raise RuntimeError("Falha no download da base pública: " + " | ".join(erros))

baixar_base()
os.environ.setdefault("MPLCONFIGDIR", str(ROOT / ".mplconfig"))
pd.set_option("display.max_columns", 30)
{"Python": platform.python_version(), "pandas": pd.__version__, "scikit-learn": sklearn.__version__, "H3": h3.__version__, "Plotly": plotly.__version__}


> **Reprodutibilidade:** o notebook não depende de ZIP, repositório remoto ou módulos auxiliares. O CSV público é conferido por SHA-256; as funções permanecem visíveis junto às respectivas etapas e as versões do ambiente são registradas ao final.
